<a href="https://colab.research.google.com/github/Lagnajit09/100x_AI_ML/blob/main/TrainingModel_MultiHeadAttention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Head Attention

### Why do we need multiple heads?

One attention head produces one set of attention weights — one perspective on how words relate. But language is complex. A single word can have multiple types of relationships simultaneously.

Take the sentence: `"The cat sat on the mat because it was tired"`

The word "it" needs to figure out:
```
Relationship 1: "it" refers to "cat" (not "mat") — coreference
Relationship 2: "it" comes after "because"       — syntactic structure  
Relationship 3: "it" connects to "tired"         — who is tired?
```
One attention head can only learn one pattern of Q-K matching. It might learn to find coreferences, but then it can't simultaneously learn syntactic patterns. That's like having one employee who can only do one job.

Multi-head attention runs several attention heads in parallel, each with its own W_Q, W_K, W_V. Each head is free to learn a different type of relationship. Then we combine their outputs at the end.
```
Single head:  ONE perspective   → limited understanding
Multi-head:   MANY perspectives → rich understanding

Head 1: "Who does this pronoun refer to?"
Head 2: "What's the verb connected to this subject?"
Head 3: "Which adjective modifies this noun?"
Head 4: "What comes next in this phrase?"
```


### The math — how d_k connects to number of heads
`d_k` is smaller than `d_model` to make room for multiple heads.

Here's exactly how:
```
d_model = 8     (each word's full embedding size)
num_heads = 2   (we want 2 perspectives)
d_k = d_model / num_heads = 8 / 2 = 4

Head 1: works in 4 dimensions with its own W_Q, W_K, W_V
Head 2: works in 4 dimensions with its own W_Q, W_K, W_V

Total dimensions used: 4 + 4 = 8 = d_model ✓
```
Each head gets a slice of the full dimensionality. The total compute stays the same as one big head — we're just splitting the work.

In real models:
```
GPT-2:  d_model=768,   12 heads, d_k = 768/12  = 64 per head
GPT-3:  d_model=12288, 96 heads, d_k = 12288/96 = 128 per head
```


## Step 1 — Let's build it

```python
# MULTI-HEAD ATTENTION
#
# One head = one perspective on word relationships
# Multiple heads = multiple perspectives running in parallel
#
# Each head has its own W_Q, W_K, W_V and learns different patterns.
# d_k = d_model / num_heads — each head works in a smaller space.
# After all heads compute attention, their outputs are concatenated
# and projected back to d_model dimensions.
#
# Flow:
# input (7, 8) → split into 2 heads → each head does attention in (7, 4)
# → concatenate outputs → (7, 8) → project back → (7, 8)
```

In [4]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        # d_k is CALCULATED, not chosen by us
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # each head's dimension

        # Each head needs its own Q, K, V projections
        # But instead of making separate nn.Linear for each head,
        # we use ONE big linear layer and then SPLIT the output
        # This is more efficient — one big matrix multiply instead of many small ones
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # (8, 8) not (8, 4)!
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)

        # After concatenating all heads, project back to d_model
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        seq_len = x.shape[0]     # number of tokens (7)

        # Step 1: Project into Q, K, V (full d_model size)
        Q = self.W_Q(x)   # (7, 8)
        K = self.W_K(x)   # (7, 8)
        V = self.W_V(x)   # (7, 8)

        # Step 2: SPLIT into multiple heads
        # Reshape from (7, 8) → (7, 2, 4) → (2, 7, 4)
        # 7 tokens, 2 heads, 4 dimensions per head
        Q = Q.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        K = K.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        V = V.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        # Now Q, K, V are each (2, 7, 4) — 2 heads, 7 tokens, 4 dims per head

        # Step 3: Attention (same formula, now per head)
        scores = Q @ K.transpose(-2, -1)       # (2, 7, 7) — each head has its own 7×7
        scaled_scores = scores / math.sqrt(self.d_k)
        attention_weights = torch.softmax(scaled_scores, dim=-1)  # (2, 7, 7)

        # Step 4: Multiply by V
        head_outputs = attention_weights @ V    # (2, 7, 4)

        # Step 5: CONCATENATE all heads back together
        # (2, 7, 4) → (7, 2, 4) → (7, 8)
        head_outputs = head_outputs.permute(1, 0, 2).contiguous().view(seq_len, self.d_model)

        # Step 6: Final projection
        output = self.W_O(head_outputs)    # (7, 8)

        return output, attention_weights

### Step 2 — Understanding the split

The key insight is the reshape and permute in Step 2:

```python
# Let's trace what happens to Q with actual shapes

# After W_Q projection:
# Q shape: (7, 8) — 7 tokens, 8 dimensions
#
# Token "i":    [a, b, c, d, e, f, g, h]   ← 8 numbers
#
# We SPLIT those 8 numbers into 2 groups of 4:
#   Head 1 gets: [a, b, c, d]   ← first 4 dimensions
#   Head 2 gets: [e, f, g, h]   ← last 4 dimensions

# .view(7, 2, 4) does the split:
# (7, 8) → (7, 2, 4)
# 7 tokens, 2 heads, 4 dims each
#
# Token "i" becomes:
#   Head 1: [a, b, c, d]
#   Head 2: [e, f, g, h]

# .permute(1, 0, 2) rearranges to put heads first:
# (7, 2, 4) → (2, 7, 4)
# 2 heads, 7 tokens, 4 dims each
#
# Now it's organized as:
#   Head 1: all 7 tokens with their first-4-dim queries
#   Head 2: all 7 tokens with their last-4-dim queries
```

### Why Put Heads First?

Imagine we have the Q tensor after splitting:
```
After .view(7, 2, 4):  shape is (7, 2, 4)
                         tokens, heads, dims

Organized as:
  Token 0: [Head1: [a,b,c,d], Head2: [e,f,g,h]]
  Token 1: [Head1: [i,j,k,l], Head2: [m,n,o,p]]
  Token 2: [Head1: [q,r,s,t], Head2: [u,v,w,x]]
  ...  
```

The data is grouped BY TOKEN. But attention needs to compare
tokens WITHIN each head.

After .permute(1, 0, 2) — heads come first:

```
After permute: shape is (2, 7, 4)
                         heads, tokens, dims

Organized as:
  Head 1: [Token0: [a,b,c,d],  Token1: [i,j,k,l],  Token2: [q,r,s,t], ...]
  Head 2: [Token0: [e,f,g,h],  Token1: [m,n,o,p],  Token2: [u,v,w,x], ...]
```
Now the data is grouped BY HEAD. Each head has all 7 tokens together.

Why does this matter? When we do Q @ K.T, PyTorch treats the first dimension as a batch. It says "do separate matrix multiplications for each item in dimension 0":

```
Q @ K.T with shape (2, 7, 4) @ (2, 4, 7):

PyTorch does:
  Batch 0 (Head 1): (7, 4) @ (4, 7) = (7, 7)  ← head 1's attention
  Batch 1 (Head 2): (7, 4) @ (4, 7) = (7, 7)  ← head 2's attention
```

Result: (2, 7, 7) — two separate attention matrices, one per head
If heads were NOT first, PyTorch would try to compare tokens across heads, which is wrong. Head 1's queries should only match with Head 1's keys, not Head 2's keys. Putting heads first makes PyTorch treat each head as an independent attention computation.

### Step 3 — Understanding the concatenation

After attention, each head produced its own output:
```
Head 1 output: (7, 4) — 7 tokens, each with 4-dim contextualized vector
Head 2 output: (7, 4) — 7 tokens, each with 4-dim contextualized vector

Combined: (2, 7, 4)

We reverse the split:
permute(1, 0, 2): (2, 7, 4) → (7, 2, 4)
view(7, 8):       (7, 2, 4) → (7, 8)

Token "i" now has: [head1_a, head1_b, head1_c, head1_d,
                    head2_a, head2_b, head2_c, head2_d]
```

Both perspectives glued together into one 8-dim vector.


### Step 4 — What's W_O and why?
After concatenation, each token has information from multiple heads just stacked side by side. W_O (the output projection) mixes them together:
```
Without W_O: [head1 info | head2 info]  — just glued, no interaction
With W_O:    head1 and head2 info BLENDED into a unified representation
```
It's like having two advisors give you separate reports. W_O is you reading both and writing a unified summary.

### Step 5 — Run It

In [5]:
# Same sentence setup
text = "I love code and I love AI"
tokens = text.lower().split()
vocab = sorted(set(tokens))
word_to_id = {word: idx for idx, word in enumerate(vocab)}
token_ids = torch.tensor([word_to_id[t] for t in tokens])

# Embedding layer (d_model=8 now, to allow 2 heads of 4 each)
d_model = 8
embedding = nn.Embedding(len(vocab), d_model)

# Create positional encoding
def create_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            denominator = 10000 ** (i / d_model)
            pe[pos, i]     = math.sin(pos / denominator)
            pe[pos, i + 1] = math.cos(pos / denominator)
    return pe

pe = create_positional_encoding(10, d_model)

# Get embeddings + positional encoding
final_embeddings = embedding(token_ids) + pe[:len(tokens)]

# Create multi-head attention (2 heads)
mha = MultiHeadAttention(d_model=8, num_heads=2)

# Run it!
output, attention_weights = mha(final_embeddings)

print("Input shape:", final_embeddings.shape)    # (7, 8)
print("Output shape:", output.shape)              # (7, 8)
print("Attention shape:", attention_weights.shape) # (2, 7, 7) — one 7×7 per head

print("\nHead 1 attention (who pays attention to whom):")
print(attention_weights[0])

print("\nHead 2 attention (DIFFERENT pattern!):")
print(attention_weights[1])

Input shape: torch.Size([7, 8])
Output shape: torch.Size([7, 8])
Attention shape: torch.Size([2, 7, 7])

Head 1 attention (who pays attention to whom):
tensor([[0.1424, 0.1390, 0.1264, 0.0935, 0.1428, 0.1773, 0.1786],
        [0.1172, 0.1526, 0.1393, 0.1148, 0.1255, 0.1720, 0.1785],
        [0.1369, 0.1321, 0.1557, 0.1410, 0.1509, 0.1251, 0.1583],
        [0.1363, 0.1432, 0.1410, 0.1583, 0.1306, 0.1353, 0.1553],
        [0.1689, 0.1155, 0.1496, 0.1097, 0.1890, 0.1227, 0.1446],
        [0.1309, 0.1209, 0.1614, 0.1012, 0.1673, 0.1292, 0.1890],
        [0.1063, 0.1410, 0.1032, 0.0777, 0.0968, 0.1987, 0.2763]],
       grad_fn=<SelectBackward0>)

Head 2 attention (DIFFERENT pattern!):
tensor([[0.1763, 0.2045, 0.1025, 0.0782, 0.1542, 0.1740, 0.1102],
        [0.1733, 0.0977, 0.1383, 0.1201, 0.1609, 0.1044, 0.2053],
        [0.1926, 0.1736, 0.0940, 0.0788, 0.1495, 0.1818, 0.1297],
        [0.2127, 0.1405, 0.0901, 0.0885, 0.1319, 0.1434, 0.1929],
        [0.2268, 0.2217, 0.0656, 0.0443, 0.1466

### Step 6 — Compare the two heads
This is the whole point — look at how the two heads develop different attention patterns:

Even with random weights, the two heads will likely show different attention patterns. In a trained model, these differences would be meaningful — one head might track grammar, another might track meaning.

In [6]:
print("=== Word 'i' (position 0) attention patterns ===")
print(f"\nHead 1: {attention_weights[0][0].detach()}")
print(f"Head 2: {attention_weights[1][0].detach()}")
print(f"\nHead 1 thinks 'i' should attend most to: {tokens[attention_weights[0][0].argmax()]}")
print(f"Head 2 thinks 'i' should attend most to: {tokens[attention_weights[1][0].argmax()]}")

=== Word 'i' (position 0) attention patterns ===

Head 1: tensor([0.1424, 0.1390, 0.1264, 0.0935, 0.1428, 0.1773, 0.1786])
Head 2: tensor([0.1763, 0.2045, 0.1025, 0.0782, 0.1542, 0.1740, 0.1102])

Head 1 thinks 'i' should attend most to: ai
Head 2 thinks 'i' should attend most to: love


We used one big nn.Linear(d_model, d_model) for W_Q and then split the output into heads, instead of creating separate nn.Linear(d_model, d_k) for each head. Both approaches give the same result. Why is the "one big matrix, then split" approach preferred?

One big multiplication is much faster on GPUs than many small ones because GPUs are designed to handle large parallel operations. Doing 12 separate small multiplications has overhead — launching each operation, moving data around — while one big one flows through the hardware smoothly.